In [2]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_openai import ChatOpenAI
from langchain_community.llms import Ollama
import re

orders = {
    "#101": "Preparing",
    "#102": "Out for delivery",
    "#103": "Delivered",
    "#104": "Cancelled"
}

queried_orders = set()

memory = ConversationBufferMemory(
    memory_key="history",
    input_key="input",
    return_messages=False
    
)

# -------------------------
# Prompt Template
# -------------------------
prompt = PromptTemplate(
    input_variables=["history", "input", "order_status"],
    template="""
You are a food delivery order-tracking assistant.

Conversation History:
{history}

Current User Message:
{input}

Order Status:
{order_status}

Answer naturally. If the order exists, provide its status.
If the order does not exist, politely say the order ID could not be found.
"""
)

# -------------------------
# LLM
# -------------------------
llm=Ollama(
    model="llama3.1",
    temperature=0
)

chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory
)

print("Food Delivery Assistant")
print("Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "quit":
        break

    # Extract order ID
    match = re.search(r"#\d+", user_input)

    if match:
        order_id = match.group()
        queried_orders.add(order_id)
    else:
        # Try to recover previous order from memory
        history = memory.load_memory_variables({})["history"]
        previous = re.findall(r"#\d+", history)

        if previous:
            order_id = previous[-1]
        else:
            order_id = None
    101


    if order_id:
        status = orders.get(order_id, "NOT_FOUND")
    else:
        status = "NO_ORDER"

    if status == "NOT_FOUND":
        order_status = f"{order_id} was not found."
    elif status == "NO_ORDER":
        order_status = "No order ID was provided."
    else:
        order_status = f"{order_id} is currently '{status}'."

    response = chain.invoke({
        "input": user_input,
        "order_status": order_status
    })

    print("Assistant:", response["text"])
    print()

print("\nSession Summary")
print("------------------------")
print("Unique orders queried:", len(queried_orders))
print("Order IDs:", sorted(queried_orders))


Food Delivery Assistant
Type 'quit' to exit.

Assistant: Your order #101 is currently being prepared by our kitchen team. We'll let you know as soon as it's ready for pickup or delivery!

Assistant: I'm happy to help you track your order! Unfortunately, I couldn't find any information on an order with the number #104. It's possible that it was cancelled or never placed. If you have any other questions or need assistance with a different order, feel free to ask!

Assistant: Your order #103 has already been delivered! We're glad you received your food safely and enjoyably. Is there anything else I can help you with?

Assistant: Your order #102 is currently out for delivery! Our driver has picked it up and is on their way to your location. You should receive your food shortly. Would you like me to provide an estimated time of arrival?


Session Summary
------------------------
Unique orders queried: 4
Order IDs: ['#101', '#102', '#103', '#104']
